In [15]:
import os
import pandas as pd
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

In [33]:
df = pd.read_csv("data/train_full.csv")

df["image_path"] = df["image_path"].str.replace("\\", "/", regex=False)
df["image_path"] = df["image_path"].str.replace(
    "data/train_data/",
    "data/train_data/train_data/",
    regex=False
)

df.head()

,id,xmin,ymin,xmax,ymax,direction,split,image_path
0,image_0,-228.877144,223.618740,-110.614879,416.358789,1,val,data/train_data/train_data/image_0.png
1,image_2,871.866846,218.142342,945.359631,423.076854,0,train,data/train_data/train_data/image_2.png
2,image_3,-224.589965,220.840143,-146.401974,409.747294,0,val,data/train_data/train_data/image_3.png
3,image_4,898.981190,215.768093,956.279097,429.133944,1,train,data/train_data/train_data/image_4.png
4,image_5,-166.975923,223.755640,-118.486304,369.332377,1,val,data/train_data/train_data/image_5.png


In [34]:
train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df = df[df["split"] == "val"].reset_index(drop=True)

print("Train:", len(train_df))
print("Val:", len(val_df))

Train: 1353
Val: 339


In [35]:
class ShadowCsvDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = row["image_path"]
        img = Image.open(img_path).convert("RGB")

        if self.transform:
            img = self.transform(img)

        bbox = torch.tensor(
            [row["xmin"], row["ymin"], row["xmax"], row["ymax"]],
            dtype=torch.float32
        )

        direction = torch.tensor(int(row["direction"]), dtype=torch.long)

        return img, bbox, direction, row["id"]

In [36]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

In [37]:
train_dataset = ShadowCsvDataset(train_df, transform=train_transform)
val_dataset = ShadowCsvDataset(val_df, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

In [38]:
class SimpleShadowModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()

        self.box_head = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Linear(256, 4)
        )

        self.dir_head = nn.Sequential(
            nn.Linear(in_features, 128),
            nn.ReLU(),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        features = self.backbone(x)
        bbox = self.box_head(features)
        direction = self.dir_head(features)
        return bbox, direction

In [39]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleShadowModel().to(device)

bbox_loss_fn = nn.SmoothL1Loss()
dir_loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

print(device)

cpu


In [40]:
def compute_iou_xyxy(pred_boxes, true_boxes):
    xA = torch.maximum(pred_boxes[:, 0], true_boxes[:, 0])
    yA = torch.maximum(pred_boxes[:, 1], true_boxes[:, 1])
    xB = torch.minimum(pred_boxes[:, 2], true_boxes[:, 2])
    yB = torch.minimum(pred_boxes[:, 3], true_boxes[:, 3])

    inter_w = torch.clamp(xB - xA, min=0)
    inter_h = torch.clamp(yB - yA, min=0)
    inter_area = inter_w * inter_h

    pred_area = torch.clamp(pred_boxes[:, 2] - pred_boxes[:, 0], min=0) * torch.clamp(pred_boxes[:, 3] - pred_boxes[:, 1], min=0)
    true_area = torch.clamp(true_boxes[:, 2] - true_boxes[:, 0], min=0) * torch.clamp(true_boxes[:, 3] - true_boxes[:, 1], min=0)

    union = pred_area + true_area - inter_area
    iou = inter_area / torch.clamp(union, min=1e-6)

    return iou

In [41]:
def train_one_epoch(model, loader, optimizer, bbox_loss_fn, dir_loss_fn, device):
    model.train()
    total_loss = 0.0

    for images, true_bbox, true_dir, _ in loader:
        images = images.to(device)
        true_bbox = true_bbox.to(device)
        true_dir = true_dir.to(device)

        optimizer.zero_grad()

        pred_bbox, pred_dir = model(images)

        bbox_loss = bbox_loss_fn(pred_bbox, true_bbox)
        dir_loss = dir_loss_fn(pred_dir, true_dir)

        loss = bbox_loss + 0.2 * dir_loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

    return total_loss / len(loader.dataset)

In [42]:
def validate(model, loader, bbox_loss_fn, dir_loss_fn, device):
    model.eval()
    total_loss = 0.0
    all_ious = []
    all_dir_correct = []

    with torch.no_grad():
        for images, true_bbox, true_dir, _ in loader:
            images = images.to(device)
            true_bbox = true_bbox.to(device)
            true_dir = true_dir.to(device)

            pred_bbox, pred_dir = model(images)

            bbox_loss = bbox_loss_fn(pred_bbox, true_bbox)
            dir_loss = dir_loss_fn(pred_dir, true_dir)
            loss = bbox_loss + 0.2 * dir_loss

            total_loss += loss.item() * images.size(0)

            ious = compute_iou_xyxy(pred_bbox, true_bbox)
            all_ious.extend(ious.cpu().numpy())

            pred_labels = pred_dir.argmax(dim=1)
            correct = (pred_labels == true_dir).float()
            all_dir_correct.extend(correct.cpu().numpy())

    return (
        total_loss / len(loader.dataset),
        float(np.mean(all_ious)),
        float(np.mean(all_dir_correct))
    )

In [43]:
print("model:", "ok" if "model" in globals() else "missing")
print("train_loader:", "ok" if "train_loader" in globals() else "missing")
print("val_loader:", "ok" if "val_loader" in globals() else "missing")
print("train_one_epoch:", "ok" if "train_one_epoch" in globals() else "missing")
print("validate:", "ok" if "validate" in globals() else "missing")
print("bbox_loss_fn:", "ok" if "bbox_loss_fn" in globals() else "missing")
print("dir_loss_fn:", "ok" if "dir_loss_fn" in globals() else "missing")
print("optimizer:", "ok" if "optimizer" in globals() else "missing")

model: ok
train_loader: ok
val_loader: ok
train_one_epoch: ok
validate: ok
bbox_loss_fn: ok
dir_loss_fn: ok
optimizer: ok


In [ ]:
num_epochs = 10

for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, optimizer, bbox_loss_fn, dir_loss_fn, device)
    val_loss, val_iou, val_dir_acc = validate(model, val_loader, bbox_loss_fn, dir_loss_fn, device)

    print(
        f"Epoch {epoch+1}/{num_epochs} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"val_iou={val_iou:.4f} | "
        f"val_dir_acc={val_dir_acc:.4f}"
    )

In [ ]:
import cv2
import random

def show_prediction(model, dataset, idx, device, pad=500):
    model.eval()

    img_tensor, true_bbox, true_dir, image_id = dataset[idx]

    with torch.no_grad():
        pred_bbox, pred_dir = model(img_tensor.unsqueeze(0).to(device))

    pred_bbox = pred_bbox.squeeze(0).cpu().numpy()
    true_bbox = true_bbox.numpy()

    row = dataset.df.iloc[idx]
    img = cv2.imread(row["image_path"])
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    h, w = img.shape[:2]
    canvas = np.full((h + 2 * pad, w + 2 * pad, 3), 235, dtype=np.uint8)
    canvas[pad:pad+h, pad:pad+w] = img

    tx1, ty1, tx2, ty2 = [int(round(v)) + pad for v in true_bbox]
    px1, py1, px2, py2 = [int(round(v)) + pad for v in pred_bbox]

    cv2.rectangle(canvas, (tx1, ty1), (tx2, ty2), (0, 255, 0), 2)
    cv2.rectangle(canvas, (px1, py1), (px2, py2), (255, 0, 0), 2)

    pred_direction = pred_dir.argmax(dim=1).item()

    plt.figure(figsize=(12, 7))
    plt.imshow(canvas)
    plt.title(f"{image_id} | true_dir={true_dir.item()} | pred_dir={pred_direction}")
    plt.axis("off")
    plt.show()